In [2]:
import os
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras

from collections import defaultdict
from datetime import datetime
from tensorflow.keras import layers

In [3]:
tf.__version__, keras.__version__

('2.20.0', '3.13.2')

### Load Data

In [4]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(f"{x_train.shape = }, {type(x_train)}")
print(f"{x_test.shape = }, {type(x_test)}")

x_train.shape = (60000, 28, 28), <class 'numpy.ndarray'>
x_test.shape = (10000, 28, 28), <class 'numpy.ndarray'>


### Data Preprocessing

In [5]:
def max_min_values(x_train, x_test):
    print(f"Train\n    Max: {x_train.max()}\n    Min: {x_train.min()}") 
    print(f"Test\n    Max: {x_test.max()}\n    Min: {x_test.min()}")  

max_min_values(x_train, x_test)

Train
    Max: 255
    Min: 0
Test
    Max: 255
    Min: 0


In [6]:
# Normalization (Scaling the range [0, 255] to [0.0, 1.0])
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

max_min_values(x_train, x_test)

Train
    Max: 1.0
    Min: 0.0
Test
    Max: 1.0
    Min: 0.0


In [7]:
# Adding channel dimension (60000, 28, 28) -> (60000, 28, 28, 1) 
print(f"Previous: {x_train.shape = }")
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)
print(f"New: {x_train.shape = }")

Previous: x_train.shape = (60000, 28, 28)
New: x_train.shape = (60000, 28, 28, 1)


In [8]:
n_classes = len(np.unique(y_train))
input_shape = x_train.shape[1:]
n_classes, input_shape

(10, (28, 28, 1))

In [9]:
y_train[:5]

array([5, 0, 4, 1, 9], dtype=uint8)

In [10]:
# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, n_classes)
y_test = keras.utils.to_categorical(y_test, n_classes)

y_train[:5]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

### Build Neural Network

In [11]:
class ImageClassifier(keras.Model):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.conv2d_1 = layers.Conv2D(32, (3, 3), activation="relu", name="Conv1")
        self.conv2d_2 = layers.Conv2D(64, (3, 3), activation="relu", name="Conv2")
        self.max_pool_1 = layers.MaxPool2D((2, 2), name="MaxPool1")
        self.max_pool_2 = layers.MaxPool2D((2, 2), name="MaxPool2")
        self.flatten = layers.Flatten(name="Flatten")
        self.dropout = layers.Dropout(0.5, name='Dropout')
        self.outputs = layers.Dense(n_classes, activation="softmax", name="Output")

    def call(self, inputs):
        x = self.conv2d_1(inputs)
        x = self.max_pool_1(x)
        x = self.conv2d_2(x)
        x = self.max_pool_2(x)
        x = self.flatten(x)
        x = self.dropout(x)
        outputs = self.outputs(x)
        return outputs
    
    def build_graph(self):
        x = keras.Input(shape=input_shape, name="Input")
        return keras.Model(inputs=x, outputs=self.call(x), name="ImageClassifier")
    
model = ImageClassifier()
model.build_graph().summary()

Model: "ImageClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool1 (MaxPooling2D)         │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool2 (MaxPooling2D)         │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

### Train

In [12]:
batch_size = 128
epochs = 15
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 19s 37ms/step - accuracy: 0.9230 - loss: 0.2830 - val_accuracy: 0.9777 - val_loss: 0.0833
Epoch 2/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step - accuracy: 0.9769 - loss: 0.0758 - val_accuracy: 0.9830 - val_loss: 0.0616
Epoch 3/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 14s 34ms/step - accuracy: 0.9829 - loss: 0.0548 - val_accuracy: 0.9877 - val_loss: 0.0442
Epoch 4/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 17s 39ms/step - accuracy: 0.9861 - loss: 0.0447 - val_accuracy: 0.9888 - val_loss: 0.0398
Epoch 5/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 21s 40ms/step - accuracy: 0.9884 - loss: 0.0368 - val_accuracy: 0.9893 - val_loss: 0.0381
Epoch 6/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 20s 38ms/step - accuracy: 0.9899 - loss: 0.0330 - val_accuracy: 0.9887 - val_loss: 0.0382
Epoch 7/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 16s 39ms/step - accuracy: 0.9915 - loss: 0.0281 - val_accuracy: 0.9908 - val_loss: 0.0356
Epoch 8/15
422/422 ━━━━━━━━━━━━━━━━━━━━ 16s 38ms/step - accuracy: 0.9920 - loss: 0.0251 - 

### Evaluate 

In [13]:
score = model.evaluate(x_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Test loss: 0.033110905438661575
Test accuracy: 0.9904999732971191


## Gradient Visualization Part

### Tensorflow log directory

In [ ]:
# model number suffix identification
from glob import glob

root_dir = "logs/gradient_monitor"
log_dirs = glob("model-*", root_dir=root_dir)

suffix = max([
    int(tail) for log_dir in log_dirs 
    if (tail:=log_dir.rsplit("-", 1)[-1]).isdigit()
] + [0])

new_suffix = suffix + 1

log_dir = os.path.join(
    root_dir,
    f"model-{new_suffix}"
)

os.makedirs(log_dir, exist_ok=True)

summary_writer = tf.summary.create_file_writer(log_dir)

1

In [ ]:
from typing import Any


class GradientMonitor:
    def __init__(self, writer) -> None:
        self.history2 = defaultdict(list)
        self.writer = writer
        self.step = 0

    def log(self, gradients, variables):
        with self.writer.as_default():
            global_norms = []

            for grad, var in zip(gradients, variables):
                if grad is None:
                    continue

                grad_norm = tf.norm(grad)
                weight_norm = tf.norm(var)
                ratio = grad_norm / (weight_norm + 1e-12)

                global_norms.append(grad_norm)

                # Per-layer scalar logging
                tf.summary.scalar(
                    f"grad_norm/{var.name}", grad_norm, step=self.step
                )
                tf.summary.scalar(
                    f"grad_ratio/{var.name}", ratio, step=self.step
                )
                tf.summary.scalar(
                    "grad_mean", tf.reduce_mean(grad), step=self.step
                )
                tf.summary.scalar(
                    "grad_std", tf.math.reduce_std(grad), step=self.step
                )
                tf.summary.scalar(
                    "grad_max", tf.reduce_max(grad), step=self.step
                )
                tf.summary.scalar(
                    "grad_min", tf.reduce_min(grad), step=self.step
                )
                # Optional histogram
                tf.summary.histogram(
                    f"grad_hist/{var.name}", grad, step=self.step
                )

            # Global scalar logging
            if global_norms:
                tf.summary.scalar(
                    "global_grad_norm_mean",
                    tf.reduce_mean(global_norms),
                    step=self.step
                )

        self.step += 1

    def capture(self, gradients, variables):
        for grad, var in zip(gradients, variables):
            if grad is None:
                continue

            grad_norm = tf.norm(grad)
            weight_norm = tf.norm(var)
            grad_weight_ratio = grad_norm / (weight_norm + 1e-12)

            grad_mean = tf.reduce_mean(grad)
            grad_std = tf.math.reduce_std(grad)

            self.history2[var.name].append({
                "grad_norm": grad_norm,
                "weight_norm": weight_norm,
                "grad_weight_ratio": grad_weight_ratio,
                "grad_mean": grad_mean,
                "grad_std": grad_std,
            })


class GradientMonitoredImageClassifier(keras.Model):
    def __init__(self, monitor, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.monitor = monitor

        self.conv2d_1 = layers.Conv2D(32, (3, 3), activation="relu", name="Conv1")
        self.conv2d_2 = layers.Conv2D(64, (3, 3), activation="relu", name="Conv2")
        self.max_pool_1 = layers.MaxPool2D((2, 2), name="MaxPool1")
        self.max_pool_2 = layers.MaxPool2D((2, 2), name="MaxPool2")
        self.flatten = layers.Flatten(name="Flatten")
        self.dropout = layers.Dropout(0.5, name='Dropout')
        self.outputs = layers.Dense(n_classes, activation="softmax", name="Output")

    def call(self, inputs, training=False):
        x = self.conv2d_1(inputs)
        x = self.max_pool_1(x)
        x = self.conv2d_2(x)
        x = self.max_pool_2(x)
        x = self.flatten(x)
        x = self.dropout(x, training=training)
        outputs = self.outputs(x)
        return outputs
    
    def train_step(self, data):
        x, y = data

        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            loss = self.compiled_loss(y, y_pred)
        
        gradients = tape.gradient(loss, self.trainable_variables)

        # capture gradient of the train step before updating
        if (
            self.monitor is not None and isinstance(self.monitor, GradientMonitor)
        ):
            # self.monitor.capture(gradients, self.trainable_variables)
            self.monitor.log(gradients, self.trainable_variables)
        
        self.optimizer.apply_gradients(zip(gradients, self.trainable_variables))

        self.compiled_metrics.update_state(y, y_pred)

        return {m.name: m.result() for m in self.metrics}
    
    def build_graph(self):
        x = keras.Input(shape=input_shape, name="Input")
        return keras.Model(inputs=x, outputs=self.call(x), name="ImageClassifier")
    
model2 = GradientMonitoredImageClassifier(GradientMonitor(summary_writer))
model2.build_graph().summary()

Model: "ImageClassifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input (InputLayer)              │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv1 (Conv2D)                  │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool1 (MaxPooling2D)         │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Conv2 (Conv2D)                  │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MaxPool2 (MaxPooling2D)         │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Dropout (Dropout)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output (Dense)                  │ (None, 10)             │        16,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Tensorboard callback to run once every epoch
tensorboard_callback = tf.keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=0
)

model2.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)

history2 = model2.fit(
    x_train, y_train, batch_size=batch_size, 
    epochs=2, validation_split=0.1,
    callbacks=[tensorboard_callback]
)

Epoch 1/2


c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\keras\src\backend\tensorflow\trainer.py:695: UserWarning: `model.compiled_loss()` is deprecated. Instead, use `model.compute_loss(x, y, y_pred, sample_weight, training)`.
  warnings.warn(
c:\Users\rajba\miniconda3\envs\uwl\Lib\site-packages\keras\src\backend\tensorflow\trainer.py:670: UserWarning: `model.compiled_metrics()` is deprecated. Instead, use e.g.:
```
for metric in self.metrics:
    metric.update_state(y, y_pred)
```

  return self._compiled_metrics_update_state(


422/422 ━━━━━━━━━━━━━━━━━━━━ 24s 43ms/step - accuracy: 0.8897 - loss: 0.1000 - val_accuracy: 0.9777 - val_loss: 0.0868
Epoch 2/2
422/422 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.9642 - loss: 0.1000 - val_accuracy: 0.9840 - val_loss: 0.0612


In [17]:
model2.monitor.history2

defaultdict(list, {})

In [18]:
history2.history

{'accuracy': [0.8896852135658264, 0.9642037153244019],
 'loss': [0.09999983757734299, 0.09999983757734299],
 'val_accuracy': [0.9776666760444641, 0.984000027179718],
 'val_loss': [0.08684104681015015, 0.06116200238466263]}